# Step 14 — Multi-Agent (Hierarchical)

🇬🇧 **English** (this notebook)

**Optional.** [Step 13](step_13_multi_agent_seq.ipynb) is the last required step — your final submission is due after that notebook, not this one. This one revisits the same two agents under `process=Process.hierarchical`: instead of `agent=...` fixing which agent does which task, a **manager** decides that at runtime.

## Learning objective

By the end of this notebook, you will:

- Understand how `process=Process.hierarchical` differs from `sequential`: task order is still fixed, but *which* agent handles each task is a manager's runtime decision, not something wired into the code
- Have run the same two agents from Step 13 under a manager, via `manager_llm`
- Be able to judge, on separate axes, whether that runtime flexibility actually changed anything — or just added cost

## Prerequisites

- [Step 13 — Multi-Agent (Sequential)](step_13_multi_agent_seq.ipynb) completed — this notebook reuses its Researcher and Analyst agents unchanged, and you'll want that notebook's printed output open to compare against
- The same `.env` setup as the previous steps

## Background

Step 13's `Crew` is a fixed pipeline: `agent=...` on every `Task` decides who does what, before the crew ever runs. `Process.hierarchical` is CrewAI's version of a different coordination pattern from the same multi-agent literature — a manager role that reasons about *who* should do a piece of work, rather than that being decided in advance:

> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

## How this works

In Step 13, each `Task` is hard-wired to one `Agent` via `agent=...`, and `crew.kickoff()` runs them in the order they appear in `tasks=[...]`. `Process.hierarchical` removes that fixed wiring: the tasks below don't get an `agent=` at all. Instead, a **manager** — an `Agent` CrewAI builds for you from `manager_llm` (or your own `Agent` passed as `manager_agent`) — reads each task and decides, at that moment, which of the crew's agents is best suited, then delegates to them via a tool call.

Two things worth being precise about, since "hierarchical" invites the wrong mental model:

- **Task order still isn't dynamic.** The `for` loop over `tasks=[...]` doesn't change — task 1 still runs before task 2. What's dynamic is *who* does each task, decided fresh every run, not *when* it happens.
- **The manager is not one of your agents.** It's a separate role that does no task work itself — it only reads tasks and delegates. CrewAI enforces this: a `manager_agent` may not also appear in `agents=[...]`, and it may not be given `tools=` (only the agents it delegates to can have tools).

The `researcher` and `analyst` below are the exact same `role`/`goal`/`backstory` as Step 13 — redefined here so this notebook runs standalone, not because anything about either agent changed. `tracing=True` is set here too, same as Step 13, so you can open both runs' trace links side by side at `app.crewai.com`.

In [ ]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same as Step 13, unchanged ─────────────────────────────────────
researcher = Agent(
    role=f"{topic} Senior Data Researcher",
    goal=f"Uncover cutting-edge developments in {topic}",
    backstory=(
        f"You're a seasoned researcher with a knack for uncovering the latest "
        f"developments in {topic}. Known for your ability to find the most relevant "
        f"information and present it in a clear and concise manner."
    ),
    verbose=True,
)

# ── Agent 2: Analyst — same as Step 13, unchanged ───────────────────────────────────
analyst = Agent(
    role=f"{topic} Reporting Analyst",
    goal=f"Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You're a meticulous analyst with a keen eye for detail. You're known for "
        "your ability to turn complex data into clear and concise reports, making "
        "it easy for others to understand and act on the information you provide."
    ),
    verbose=True,
)

# ── Same two tasks as Step 13, but neither gets an `agent=` — the manager decides who does what ──
research_task = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
)

analysis_task = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
)

# ── Crew — same two agents as coworkers, orchestrated by a manager instead of fixed code ──
crew = Crew(
    agents=[researcher, analyst],  # the manager's coworkers — the manager itself is not one of them
    tasks=[research_task, analysis_task],
    process=Process.hierarchical,
    manager_llm=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"),
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes — open it
    # next to Step 13's trace link to compare delegation paths directly.
    tracing=True,
)

# ── Process — kick off the crew ─────────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Research task, as delegated by the manager ===")
print(research_task.output.raw)
print("\n=== Analysis task, as delegated by the manager ===")
print(analysis_task.output.raw)

## Your task

1. Run the cell, then judge it against your Step 13 (sequential) run on three separate axes — they don't move together:
   - **Content** — put this notebook's Analyst report next to Step 13's. Different wording aside, is there an actual difference in what's covered or recommended, or are they substantively the same?
   - **Delegation path** — open both trace links (`app.crewai.com`, no signup needed) and compare timelines, or scan this cell's verbose log for "Delegate work to coworker" / "Ask question to coworker" tool calls. Did the manager route to the Researcher for the first task and the Analyst for the second — matching Step 13's fixed order — or something else?
   - **Cost & latency** — same two trace timelines: count LLM calls and total run time. `Process.hierarchical` always adds at least one extra call (the manager's own reasoning) on top of whichever agent it delegates to.

   For two agents this cleanly separated, the honest answer is usually: same content, same delegation order, at a higher cost — see Shortcomings below for when that stops being true.

2. Swap in the same custom topic/agents you used for your own Step 13 submission, and rerun. Does the manager's delegation still match the order you'd have hard-coded yourself?

## Shortcomings

`Process.hierarchical` fixes *who* handles each task, not *when* — the task order in `tasks=[...]` is still fixed, and it costs more: every task now takes an extra LLM call (the manager's own reasoning) on top of whichever agent it delegates to, and the manager's choice of delegate isn't guaranteed to be consistent between runs. For two agents with clearly non-overlapping roles like these, that extra machinery mostly buys you nothing — it starts to pay off once you have enough agents, or similar-enough roles, that hard-coding `agent=...` on every task stops being obvious.

This crew doesn't use any of the tools/MCP/RAG grounding from Steps 10–12 either — combining multi-agent orchestration with a grounded, tool-using Researcher is a natural next experiment, just not one this notebook walks you through.

## Resources for further reading

- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)
- [CrewAI `Process` concept docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs. `hierarchical`

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. Add it to `agents=[...]` and give it a `Task` in `tasks=[...]`, but don't pin it with `agent=...` — let the manager decide on its own whether and when to bring it in, rather than wiring it into a fixed spot in the pipeline like you'd have to for Step 13's sequential version. Does the output meaningfully change? If it doesn't, ask yourself why.

---

This notebook is optional and not part of your graded submission — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for what's actually required.